[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# Expresiones en *Polars*.

## ¿Qué es el sistema de expresiones?

La diferencia conceptual más importante entre *Pandas* y *Polars* es el **modelo de expresiones**.

* **Pandas (imperativo):** las operaciones se ejecutan de inmediato, columna por columna.
* **Polars (expresivo):** las operaciones son *descripciones* de transformaciones que *Polars* puede optimizar, paralelizar y componer antes de ejecutar.

```python
# Pandas — imperativo
df["total"] = df["precio"] * df["cantidad"]

# Polars — expresivo
df.with_columns(
    (pl.col("precio") * pl.col("cantidad")).alias("total")
)
```

Las expresiones se evalúan dentro de `select()`, `with_columns()`, `filter()` o `group_by().agg()`. Este notebook cubre los bloques más importantes del sistema de expresiones que encontrarás en código real generado por herramientas modernas.

https://docs.pola.rs/user-guide/expressions/

In [ ]:
import polars as pl
import numpy as np
from datetime import date, datetime

print(f"Polars {pl.__version__}")

## El sistema de expresiones

### Funciones fundamentales

| Función | Descripción |
| :--- | :--- |
| `pl.col("nombre")` | Referencia a una columna por nombre |
| `pl.col("*")` / `pl.all()` | Todas las columnas |
| `pl.exclude("col")` | Todas las columnas excepto las indicadas |
| `pl.lit(valor)` | Valor literal (constante) |
| `pl.first()` / `pl.last()` | Primera / última columna |

Las expresiones son **componibles**: se encadenan operaciones como `.alias()`, `.cast()`, `.round()`, `.abs()`, `.fill_null()`, etc.

https://docs.pola.rs/api/python/stable/reference/expressions/index.html

**Ejemplos:**

* La siguiente celda crea el `DataFrame` de ventas que se usará en todos los ejemplos de esta sección.

In [ ]:
df = pl.DataFrame({
    "id":         [1, 2, 3, 4, 5, 6],
    "producto":   ["  Widget A  ", "Gadget B", "  Widget C", "Gadget A  ", "Widget B", "Gadget C"],
    "categoria":  ["electronica", "hogar", "electronica", "hogar", "electronica", "hogar"],
    "precio":     [100.0, 250.0, 150.0, 300.0, 120.0, 80.0],
    "cantidad":   [10, 5, 8, 3, 15, 20],
    "fecha_alta": ["2023-01-15", "2023-03-22", "2022-11-08", "2024-01-01", "2023-07-30", "2022-06-15"],
})
print(df)

* La siguiente celda demuestra `pl.col()` con aritmética y `.alias()`. Nota la diferencia entre `select()` (devuelve *solo* las columnas indicadas) y `with_columns()` (agrega columnas manteniendo el `DataFrame` original).

In [ ]:
# select(): retorna SOLO las columnas especificadas
resultado_select = df.select([
    pl.col("producto"),
    pl.col("precio"),
    (pl.col("precio") * pl.col("cantidad")).alias("total"),
    (pl.col("precio") * 1.16).round(2).alias("precio_con_iva"),
])
print("select():")
print(resultado_select)

# with_columns(): AGREGA columnas al DataFrame existente
resultado_with = df.with_columns([
    (pl.col("precio") * pl.col("cantidad")).alias("total"),
    (pl.col("precio") * 1.16).round(2).alias("precio_con_iva"),
])
print("\nwith_columns():")
print(resultado_with)

* La siguiente celda ilustra `pl.lit()` para insertar constantes y `pl.exclude()` para seleccionar todas las columnas excepto las indicadas.

In [ ]:
# pl.lit(): valor literal como expresión
print("Con columna de moneda constante:")
print(
    df.with_columns(pl.lit("MXN").alias("moneda"))
      .select(["producto", "precio", "moneda"])
)

# pl.exclude(): todas las columnas excepto las indicadas
print("\nSin columnas 'id' y 'fecha_alta':")
print(df.select(pl.exclude("id", "fecha_alta")))

## Expresiones condicionales: `pl.when().then().otherwise()`

Equivalente a `np.where()` de *NumPy* o `CASE WHEN` de *SQL*, pero composable dentro de cualquier expresión de *Polars*.

```
pl.when(<condición>)
  .then(<valor_si_verdadero>)
  .otherwise(<valor_si_falso>)
```

Donde:
* `<condición>` es una expresión que evalúa a `True` o `False` (ej. `pl.col("cantidad") > 10`).
* `<valor_si_verdadero>` es el valor o expresión que se devuelve si la condición es `True`.
* `<valor_si_falso>` es el valor o expresión que se devuelve si la condición es `False`.

Se pueden encadenar múltiples condiciones:

```python
pl.when(<cond_1>).then(val_1)
  .when(cond_2).then(val_2)
  .otherwise(val_3)
```
Donde:

* `<cond_1>`, `<cond_2>`, etc. son condiciones evaluadas en orden.
* `val_1`, `val_2`, etc. son los valores o expresiones correspondientes a cada condición.
* `val_3` es el valor o expresión devuelto si ninguna condición es `True`.

https://docs.pola.rs/api/python/stable/reference/expressions/whenthen.html

**Ejemplos:**

* La siguiente celda crea la columna `nivel_precio` categorizando el precio en tres rangos usando `pl.when().then().when().then().otherwise()`.

In [ ]:
resultado = df.with_columns(
    pl.when(pl.col("precio") >= 200)
      .then(pl.lit("alto"))
      .when(pl.col("precio") >= 100)
      .then(pl.lit("medio"))
      .otherwise(pl.lit("bajo"))
      .alias("nivel_precio")
)
print(resultado.select(["producto", "precio", "nivel_precio"]))

* La siguiente celda combina `pl.when()` con expresiones aritméticas: `then()` y `otherwise()` aceptan cualquier expresión, no solo literales. Se calculan precio con descuento diferenciado y el ahorro correspondiente.

In [ ]:
resultado = df.with_columns([
    pl.when(pl.col("categoria") == "electronica")
      .then(pl.col("precio") * 0.90)    # 10 % de descuento
      .otherwise(pl.col("precio") * 0.95)  # 5 % en otras categorías
      .round(2)
      .alias("precio_oferta"),

    (pl.col("precio") -
     pl.when(pl.col("categoria") == "electronica")
       .then(pl.col("precio") * 0.10)
       .otherwise(pl.col("precio") * 0.05)
    ).round(2).alias("ahorro"),
])
print(resultado.select(["producto", "categoria", "precio", "precio_oferta", "ahorro"]))

## Tipos de datos en *Polars*

*Polars* tiene un sistema de tipos rico y explícito basado en *Apache Arrow*:

| Categoría | Tipos |
| :--- | :--- |
| **Enteros** | `Int8`, `Int16`, `Int32`, `Int64` · `UInt8`…`UInt64` (sin signo) |
| **Flotantes** | `Float32`, `Float64` |
| **Texto** | `String` (antes `Utf8`) |
| **Booleano** | `Boolean` |
| **Categórico** | `Categorical` |
| **Fechas** | `Date`, `Datetime`, `Duration`, `Time` |
| **Anidados** | `List(tipo)`, `Array(tipo, ancho)`, `Struct` |
| **Binario** | `Binary` |

A diferencia de *Pandas*, *Polars* **nunca convierte tipos en silencio** — si la conversión no es posible, lanza un error de inmediato.

https://docs.pola.rs/api/python/stable/reference/datatypes.html

**Ejemplos:**

* La siguiente celda crea un `DataFrame` con columnas de distintos tipos y muestra el schema resultante para observar la inferencia automática de tipos.

In [ ]:
df_tipos = pl.DataFrame({
    "entero":   [1, 2, 3],
    "flotante": [1.5, 2.5, 3.5],
    "texto":    ["a", "b", "c"],
    "booleano": [True, False, True],
    "fecha":    [date(2024, 1, 1), date(2024, 6, 15), date(2024, 12, 31)],
    "lista":    [[1, 2], [3, 4, 5], [6]],
})

print("Schema:")
for nombre, tipo in df_tipos.schema.items():
    print(f"  {nombre:10s} → {tipo}")
print()
print(df_tipos)

* La siguiente celda convierte columnas entre tipos con `.cast()`. *Polars* lanza un error si la conversión no es posible, a diferencia de *Pandas*, que puede producir `NaN` silenciosamente.

In [ ]:
resultado = df.with_columns([
    pl.col("precio").cast(pl.Int64).alias("precio_int"),
    pl.col("cantidad").cast(pl.Float32).alias("cantidad_float"),
    pl.col("id").cast(pl.String).alias("id_texto"),
])
print(resultado.select(["precio", "precio_int", "cantidad", "cantidad_float", "id", "id_texto"]))
print()
print("Tipos resultantes:")
for col in ["precio_int", "cantidad_float", "id_texto"]:
    print(f"  {col:20s} → {resultado.schema[col]}")

## Tipo `Categorical`

`pl.Categorical` es el equivalente a `"category"` de *Pandas*, pero más eficiente. Ideal para columnas de texto con **baja cardinalidad** (pocos valores únicos repetidos muchas veces: países, categorías, estados de proceso).

*Polars* almacena cada valor único una sola vez y usa enteros internamente, reduciendo el uso de memoria de forma significativa.

https://docs.pola.rs/api/python/stable/reference/datatypes.html#polars.Categorical

**Ejemplos:**

* La siguiente celda compara el consumo de memoria de la misma columna almacenada como `String` vs `Categorical` sobre un millón de filas.

In [ ]:
df_string = pl.DataFrame({"cat": ["electronica", "hogar"] * 500_000})
df_categ  = df_string.with_columns(pl.col("cat").cast(pl.Categorical))

mem_string = df_string.estimated_size("mb")
mem_categ  = df_categ.estimated_size("mb")

print(f"String:      {mem_string:.2f} MB")
print(f"Categorical: {mem_categ:.2f} MB")
print(f"Reducción:   {(1 - mem_categ / mem_string) * 100:.1f} %")

# Uso en el DataFrame de ejemplo
df_cat = df.with_columns(pl.col("categoria").cast(pl.Categorical))
print(f"\nSchema con Categorical: {df_cat.schema['categoria']}")
print(df_cat.select(["producto", "categoria"]))

## Namespace `.str`

El acceso `.str` expone métodos de manipulación de texto. La nomenclatura difiere ligeramente de *Pandas*:

| Pandas | Polars |
| :--- | :--- |
| `.str.lower()` | `.str.to_lowercase()` |
| `.str.upper()` | `.str.to_uppercase()` |
| `.str.len()` | `.str.len_chars()` |
| `.str.contains(pat)` | `.str.contains(pat)` |
| `.str.startswith(s)` | `.str.starts_with(s)` |
| `.str.endswith(s)` | `.str.ends_with(s)` |
| `.str.strip()` | `.str.strip_chars()` |
| `.str.replace(p, r, n=1)` | `.str.replace(p, r)` |
| `.str.replace(p, r)` (all) | `.str.replace_all(p, r)` |
| `.str.extract(pat)` | `.str.extract(pat, group_index=1)` |
| `.str.split(sep)` | `.str.split(by=sep)` → `List` |

https://docs.pola.rs/api/python/stable/reference/expressions/string.html

**Ejemplos:**

* La siguiente celda crea un `DataFrame` de empleados con problemas típicos de limpieza: espacios en blanco, capitalización inconsistente y datos de contacto estructurados.

In [ ]:
empleados = pl.DataFrame({
    "id":       [1, 2, 3, 4, 5],
    "nombre":   ["  Ana García ", "CARLOS MENDOZA", "sofía lópez  ", "MIGUEL TORRES", "  Lucía Herrera"],
    "email":    ["ana.garcia@empresa.com", "carlos.mendoza@empresa.com",
                 "sofia.lopez@empresa.com", "miguel.torres@empresa.com",
                 "lucia.herrera@empresa.com"],
    "puesto":   ["Analista Senior", "Desarrollador Jr", "Analista Jr",
                 "Desarrollador Senior", "Analista Senior"],
    "telefono": ["55-1234-5678", "55-8765-4321", "55-1111-2222",
                 "55-3333-4444", "55-5555-6666"],
})
print(empleados)

* La siguiente celda limpia `nombre` con `strip_chars()` y `to_lowercase()`, detecta empleados Senior con `contains()` y calcula la longitud del nombre con `len_chars()`.

In [ ]:
resultado = empleados.with_columns([
    # Limpiar espacios y normalizar a minúsculas
    pl.col("nombre").str.strip_chars().str.to_lowercase().alias("nombre_limpio"),

    # Detectar nivel
    pl.col("puesto").str.contains("Senior").alias("es_senior"),

    # Longitud del nombre limpio
    pl.col("nombre").str.strip_chars().str.len_chars().alias("longitud_nombre"),

    # Filtrar sólo analistas
    pl.col("puesto").str.starts_with("Analista").alias("es_analista"),
])
print(resultado.select(["nombre", "nombre_limpio", "es_senior", "es_analista", "longitud_nombre"]))

* La siguiente celda divide el email con `str.split()` (devuelve `List[String]`) y extrae el dominio con `list.get()`. También reformatea el teléfono con `str.replace_all()`.

In [ ]:
resultado = empleados.with_columns([
    # split() → List[String]; list.get(1) extrae el segundo elemento
    pl.col("email").str.split("@").list.get(1).alias("dominio"),

    # Reemplazar todos los guiones del teléfono
    pl.col("telefono").str.replace_all("-", " ").alias("telefono_formateado"),

    # Reemplazar primera ocurrencia solamente
    pl.col("puesto").str.replace("or", "OR").alias("puesto_mod"),
])
print(resultado.select(["email", "dominio", "telefono", "telefono_formateado"]))
print()
print(resultado.select(["puesto", "puesto_mod"]))

* La siguiente celda usa `str.extract()` con grupos de captura para extraer el tipo de producto y su identificador alfabético de los nombres en el `DataFrame` original.

In [ ]:
# str.extract(patrón, group_index=N) — extrae el N-ésimo grupo de captura
resultado = df.with_columns([
    pl.col("producto").str.strip_chars().str.extract(r"(Widget|Gadget)", group_index=1).alias("tipo"),
    pl.col("producto").str.strip_chars().str.extract(r"[A-Z][a-z]+ ([A-Z])", group_index=1).alias("codigo"),
])
print(resultado.select(["producto", "tipo", "codigo"]))

## Namespace `.dt`

El acceso `.dt` en columnas de tipo `Date` o `Datetime` expone métodos de extracción y operación temporal:

| Pandas | Polars |
| :--- | :--- |
| `.dt.year` | `.dt.year()` |
| `.dt.month` | `.dt.month()` |
| `.dt.day` | `.dt.day()` |
| `.dt.dayofweek` | `.dt.weekday()` (0 = lunes) |
| `.dt.strftime(fmt)` | `.dt.strftime(fmt)` |
| `pd.to_datetime(col, format=...)` | `pl.col(col).str.to_date(format=...)` |
| Suma de `timedelta` | Suma de `pl.duration(days=N)` |

https://docs.pola.rs/api/python/stable/reference/expressions/temporal.html

**Ejemplos:**

* La siguiente celda crea un `DataFrame` de ventas con fechas en formato texto y las convierte al tipo `pl.Date` con `str.to_date()`.

In [ ]:
ventas = pl.DataFrame({
    "id":           list(range(1, 9)),
    "fecha_venta":  ["2024-01-15", "2024-02-03", "2024-03-22", "2024-04-10",
                     "2024-05-05", "2024-06-18", "2024-07-30", "2024-08-12"],
    "fecha_envio":  ["2024-01-18", "2024-02-07", "2024-03-25", "2024-04-14",
                     "2024-05-09", "2024-06-21", "2024-08-02", "2024-08-16"],
    "monto":        [1200.0, 450.0, 3400.0, 780.0, 2100.0, 560.0, 4500.0, 890.0],
    "region":       ["Norte", "Sur", "Norte", "Centro", "Sur", "Centro", "Norte", "Sur"],
})

# Parsear texto → pl.Date
ventas = ventas.with_columns([
    pl.col("fecha_venta").str.to_date(format="%Y-%m-%d"),
    pl.col("fecha_envio").str.to_date(format="%Y-%m-%d"),
])
print(ventas)
print(f"\nSchema: {ventas.schema}")

* La siguiente celda extrae componentes de fecha con `.dt.year()`, `.dt.month()`, `.dt.day()`, `.dt.weekday()` y formatea la fecha como texto con `.dt.strftime()`. También filtra las ventas del primer trimestre.

In [ ]:
resultado = ventas.with_columns([
    pl.col("fecha_venta").dt.year().alias("año"),
    pl.col("fecha_venta").dt.month().alias("mes"),
    pl.col("fecha_venta").dt.day().alias("dia"),
    pl.col("fecha_venta").dt.weekday().alias("dia_semana"),  # 0=lunes, 6=domingo
    pl.col("fecha_venta").dt.strftime("%B %Y").alias("periodo"),
])
print(resultado.select(["fecha_venta", "año", "mes", "dia", "dia_semana", "periodo"]))

# Filtrar ventas de Q1
print("\nVentas Q1 (ene–mar):")
print(
    ventas.filter(pl.col("fecha_venta").dt.month().is_between(1, 3))
          .select(["fecha_venta", "monto", "region"])
)

* La siguiente celda calcula la diferencia en días entre dos fechas con `.dt.total_days()` y añade días a una fecha con `pl.duration(days=N)`. Finalmente, agrupa las ventas por mes con `group_by_dynamic()`.

In [ ]:
resultado = ventas.with_columns([
    # Días entre fecha de venta y envío
    (pl.col("fecha_envio") - pl.col("fecha_venta")).dt.total_days().alias("dias_transito"),

    # Fecha de seguimiento: 30 días después del envío
    (pl.col("fecha_envio") + pl.duration(days=30)).alias("fecha_seguimiento"),
])
print(resultado.select(["fecha_venta", "fecha_envio", "dias_transito", "fecha_seguimiento"]))

# Ventas totales por mes (group_by_dynamic)
print("\nTotal por mes:")
print(
    ventas.sort("fecha_venta")
          .group_by_dynamic("fecha_venta", every="1mo")
          .agg(pl.col("monto").sum().alias("total_mes"))
)

## Guía de traducción: *Pandas* → *Polars*

### Equivalencias clave

| Operación | Pandas | Polars |
| :--- | :--- | :--- |
| Seleccionar columnas | `df[["a", "b"]]` | `df.select(["a", "b"])` |
| Agregar columna | `df.assign(c=expr)` | `df.with_columns(expr.alias("c"))` |
| Filtrar filas | `df[df["col"] > 5]` | `df.filter(pl.col("col") > 5)` |
| Agrupar | `df.groupby("g").agg(...)` | `df.group_by("g").agg(...)` |
| Referencia a columna | `df["col"]` | `pl.col("col")` |
| Valor constante | `assign(c=1)` | `with_columns(pl.lit(1).alias("c"))` |
| Condicional | `np.where(cond, a, b)` | `pl.when(cond).then(a).otherwise(b)` |
| Texto minúsculas | `.str.lower()` | `.str.to_lowercase()` |
| Longitud de texto | `.str.len()` | `.str.len_chars()` |
| Parsear fecha | `pd.to_datetime(col, format=...)` | `pl.col(col).str.to_date(format=...)` |
| Componente año | `.dt.year` | `.dt.year()` |
| Rellenar nulos | `.fillna(0)` | `.fill_null(0)` |
| Tipo categórico | `.astype("category")` | `.cast(pl.Categorical)` |

https://docs.pola.rs/user-guide/migration/pandas/

**Ejemplo:**

* La siguiente celda ejecuta el mismo pipeline de análisis en *Pandas* y *Polars*: agregar columna calculada, clasificar con condicional, filtrar y agrupar.

In [ ]:
import pandas as pd

# ── Pipeline Pandas ──────────────────────────────────────────────────────────
df_pd = df.to_pandas()

resultado_pandas = (
    df_pd
    .assign(
        total=df_pd["precio"] * df_pd["cantidad"],
        nivel=pd.cut(df_pd["precio"], bins=[0, 100, 200, 1_000],
                     labels=["bajo", "medio", "alto"])
    )
    .query("cantidad > 5")
    .groupby("categoria", observed=True)["total"]
    .sum()
    .reset_index()
    .rename(columns={"total": "total_categoria"})
)
print("Resultado Pandas:")
print(resultado_pandas)

# ── Pipeline Polars equivalente ──────────────────────────────────────────────
resultado_polars = (
    df
    .with_columns([
        (pl.col("precio") * pl.col("cantidad")).alias("total"),
        pl.when(pl.col("precio") <= 100).then(pl.lit("bajo"))
          .when(pl.col("precio") <= 200).then(pl.lit("medio"))
          .otherwise(pl.lit("alto"))
          .alias("nivel"),
    ])
    .filter(pl.col("cantidad") > 5)
    .group_by("categoria")
    .agg(pl.col("total").sum().alias("total_categoria"))
    .sort("categoria")
)
print("\nResultado Polars:")
print(resultado_polars)

## Resumen.

Las expresiones de *Polars* proporcionan un sistema composable y optimizable para transformar datos:

| Concepto | Descripción |
| :--- | :--- |
| **Sistema de expresiones** | Operaciones que *Polars* optimiza y paraleliza antes de ejecutar |
| **`pl.when().then().otherwise()`** | Condicional composable equivalente a `CASE WHEN` de *SQL* |
| **Tipos de datos** | Sistema explícito basado en *Arrow*; nunca convierte tipos en silencio |
| **`Categorical`** | Tipo eficiente para columnas de texto con baja cardinalidad |
| **Namespace `.str`** | Métodos de manipulación de texto sobre columnas `String` |
| **Namespace `.dt`** | Métodos de extracción y aritmética temporal sobre columnas `Date`/`Datetime` |
| **Equivalencias *Pandas*** | Cada patrón imperativo de *Pandas* tiene un equivalente expresivo en *Polars* |

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>